# Introduction

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data**<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    pool all data per location and year for time-dependent stationary analysis<br>

---
**Workflow Summary (per location)**
1. Extract annual maxima → fit stationary & non-stationary GEV per model
2. Compute return levels & CIs per year per model
3. Compute exceedance probability & CI for thresholds of interest
4. Aggregate multi-model ensemble: mean + total spread
5. Visualization
   1. Return levels vs year (shaded CI)
   2. Probability amplification vs year (shaded CI)
   3. Multi-model ensemble bar plots for future RL
   4. Maps (mean & spread)
6. Tables:
   1. RL per T per year ± CI
   2. Probability change of historical RL

# Import Libraries

In [13]:
import sys
import random
import time
from datetime import datetime
from glob import glob
import warnings
import xarray as xr
from IPython.display import Markdown, display
from pandas import DataFrame
from scipy.stats import genextreme
from numpy import array, mean, min, max, sqrt, log
from scipy.stats import norm

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)

# Settings

In [2]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [3]:
hindcast_start = 1960
hindcast_end = 2026

In [4]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [5]:
dic_timing = {}
dic_notes_analysis = {}

In [6]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

In [7]:
print_msg = True

export_report=True
display_results = False
save_regression_summary = True

# Import data

In [ ]:
dic_timing['import data'] = {}
dic_timing['import data']['start'] = datetime.now()

ls_files = [file for file in glob(path + '*.nc')]
ls_files

In [ ]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

dic_timing['import data']['end'] = datetime.now()

# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, using `joblib` - saving ~60% (from 3min30sec down to 1min22sec)

In [ ]:
dic_timing['data correction'] = {}
dic_timing['data correction']['start'] = datetime.now()
dic_data_per_model = dbf.data_preparation(ls_files=ls_files, dic_data_per_model=dic_data_per_model)
dic_timing['data correction']['end'] = datetime.now()

In [ ]:
display(Markdown("**Data Overview**"))
display(Markdown("**Model · model shape: samples (~sim_years) | ensemble members | valid locations**"))
dic_notes_analysis['data overview'] = ut.create_data_overview(dic_data_per_model,ls_files)

display(Markdown(f"Execution time · {dic_timing['data correction']['end'] - dic_timing['data correction']['start']}sec"))

## Pool data per location cross models

from 8 models with up to 680 samples (sim_years) and two members and 3547-7216 locations, pool all data together and group per location
> Restructure multi-model ensemble by site <br>
> - from dic[model] -> shape: (sim_year, ensemble_member, location) <br>
> - to dic[location] -> shape: (sim_year, ensemble_member, model)


In [ ]:
dic_timing['data pooling'] = {}
dic_timing['data pooling']['start'] = datetime.now()
ls_notes = []

# ------------------------------------------------------------------------------------------
da_list = []
for model_name, dic_model in dic_data_per_model.items():
    da = dic_model['valid data'] 

    da_loc = dbf.sites_to_location(da)
    da_loc = da_loc.expand_dims(model=[model_name])

    da_list.append(da_loc)

combined = xr.concat(da_list, dim="model", join="outer")
dic_timing['data pooling']['end'] = datetime.now()

display(Markdown(f"\n**Overall, the combined dataset has the following dimensions**"))
message = f"""
        Overview of combined dataset
        \tFinal dimensions: {combined.dims}
        \tShape: {combined.shape}
        \tNumber of models: {combined.model.size}
        \tNumber of locations: {combined.location.size}
        """.strip()
print(message)
ls_notes.append(message)

display(Markdown(f"\n**Create and Store summary of locations for which we have no data in either of the models**"))
missing_locations = dbf.create_summary_location_w_missing_data(
    dic_data_per_model=dic_data_per_model, combined=combined, 
    dir_export='/'.join(path_export.split('/')[:-3]) + '/exploration'
    )
ls_notes.append(f'{len(missing_locations)} locations without any valid data found!')

# ------------------------------------------------------------------------------------------
dic_notes_analysis['data pooling'] = ls_notes

### Validation Check

In [ ]:
dic_timing['data validity check'] = {}
dic_timing['data validity check']['start'] = datetime.now()

# ------------------------------------------------------------------------------------------
list_model_labels = list(dic_data_per_model.keys())

model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

display(Markdown(f"\n**Overview Original dataArray**"))
data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

display(Markdown(f"\n**Overview Revised dataArray**"))
revised_dataset, lon_rev, lat_rev = ut.get_dataset_overview_for_model_at_location(
    dic_data=combined, model_nr=model_ex, lon=lon_target, lat=lat_target
    )

In [ ]:
if (
    lon_target == lon_rev
    and lat_target == lat_rev
    and data_for_model_for_location.dropna().equals(revised_dataset.dropna()
    )
):
    message = "Validity Check performed successfully"
else:
    message = "Validity Check failed"

dic_timing['data validity check']['end'] = datetime.now()
dic_notes_analysis['data pooling'].append(message + ' ' + str(dic_timing['data validity check']['end']))

## Rearrange data set to per location and crop to defined hind-cast period

Code parallelization for speed up, using `joblib` - saving ~65% (from 2min30sec down to 52sec)


In [ ]:
dic_timing['GEV approach 1'] = {}
dic_timing['GEV approach 1']['start'] = datetime.now()

# ------------------------------------------------------------------------------------------
results = dbf.data_rearrangement(combined=combined, hindcast_start=hindcast_start, hindcast_end=hindcast_end)
dic_data_per_location, df_messages = dbf.extract_location_data_and_info(results)
# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 1']['step1_end - prepared pooled data'] = datetime.now()

In [ ]:
df_messages

# Workflow GEV - Generalized Extreme Value

## OPTION1
Using all data available per location - from all years (within hind-cast period) and models.

**Run Analysis**<br>

Note, the output is stored as 
- visuals → png
- tabular data (DataFrames) → Parquet
- other objects (dicts, strings, floats) → Pickle

File structure
```results/
├─ location_1/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
├─ location_2/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
...
```

In [ ]:
en_end = 10
print(f'for now analyse {en_end} locations...')

In [ ]:
orig_stdout, orig_stderr, fh, logger, log_path = ut.initialize_logger(
    f"LOGS_GEVAnalysis_pooled_{datetime.now():%Y%m%d_%H%M%S}.log"
    )

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 1']['analysis start'] = datetime.now()
ls_notes = []

print("\n" + "="*100)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER LOCATION")
print("="*100)

# ------------------------------------------------------------------------------------------
results = {}
for en, location_item in enumerate(dic_data_per_location.items()):
    loc_id, df_prepared = location_item
    if en < en_end:
        print("\n" + "-"*70)
        print(f"Analyzing location id {loc_id} ...")

        lon_loc = df_prepared.lon.unique()[0]
        lat_loc = df_prepared.lat.unique()[0]

        print("\tLookup location info...")
        location_closest = dbf.add_location_labels(DataFrame([lon_loc, lat_loc], index=['lon', 'lat']).T)
        location_info = ' '.join(location_closest.values[0][2:])
        print(f"\t → Closest location identified: {location_info}")

        result, ls_warnings = gev.analyze_per_location(
            df_prepared, loc_id, lat_loc, lon_loc, location_info, return_periods
        )
        if any(ls_warnings):
            ls_notes.append(ls_warnings)
        
        if result is None:
            message = f"\t→ Warning! No valid GEV fit for location id {loc_id}. Skipping ..."
            print(message)
            ls_notes.append(message)
            continue
        
        if export_report and path_export: 
            export_path_site = ut.save_location_results(            
                location_id=loc_id, result_location=result, base_dir=path_export, 
                plot_period_evolution=plot_period_evolution, display_results=display_results
                )

        result['file_path_report'] = export_path_site
        results[loc_id] = result

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 1']['analysis end'] = datetime.now()

print("\n" + "="*100)
time_diff = dic_timing['GEV approach 1']['analysis end'] - dic_timing['GEV approach 1']['analysis start']
print(f"✓ ANALYSIS COMPLETED IN {time_diff}!")
print("="*100)

# ------------------------------------------------------------------------------------------
dic_notes_analysis['GEV pooled analysis'] = ls_notes
sys.stdout = orig_stdout
sys.stderr = orig_stderr
logger.removeHandler(fh)
fh.close()

10 locations require an execution time of ~ 30-50sec<br> 
Upscaling to 11022 locations, will result in an execution time of ~9-10hours!

## OPTION2
Re-run stationary GEV per year (for location parameter; scale and shape remain as globally defined)

In [ ]:
# del results
try:
    results.keys()
    print('✓ continue with available dictionary')
    
except NameError:
    print('import data from files...')
    results = ut.load_pooled_results('../output/gev_analysis/pooled/2026-02-06/')

In [ ]:
results = ut.select_allowed_locations(dic_data_per_location=results, start_loc=0, end_loc=10)

In [ ]:
dic_timing['GEV approach 2'] = {}
dic_timing['GEV approach 2']['analysis start'] = datetime.now()

# ------------------------------------------------------------------------------------------
results_extended, ls_notes_analysis = gev.execute_and_store_stat_gev_per_year(
    results=results, store_results=True, return_periods=return_periods
    )

# ------------------------------------------------------------------------------------------
for key, outer_list in ls_notes_analysis.items():
    ls_notes_analysis[key] = [inner for inner in outer_list if inner]
dic_notes_analysis['annual_statGEV'] = ls_notes_analysis
dic_timing['GEV approach 2']['analysis end'] = datetime.now()

# ------------------------------------------------------------------------------------------
time_diff = dic_timing['GEV approach 2']['analysis end'] - dic_timing['GEV approach 2']['analysis start']
print(f"Execution time for computing GEV per year: {time_diff}sec")

Execution time for 10 locations ~ 40sec<br> 
Upscaling to 11022 locations, will result in an execution time of ~12.5hours!

# Subsequent Analysis

## Regression curve for location parameter
including uncertainty given by n_obs

**NOTE**<br>
> centering the year parameter due to the following warning:<br>
*"The condition number is large, 2.37e+05. This might indicate that there are strong multicollinearity or other numerical problems."*

In [ ]:
colors_reg:list=['#333333FF', '#C88D35FF']
color_marker:str='#99E3DDFF'

In [ ]:
path_results = '../output/gev_analysis/pooled/2026-02-06/'

In [ ]:
results = ut.load_fit_results(path_results)
results = ut.select_allowed_locations(dic_data_per_location=results, start_loc=0, end_loc=10)

results.keys()

In [ ]:
results1 = []
for site_id, location_results in results.items():
        df = gev.weighted_least_square_regression_for_site_mp(site_id,location_results)
        results1.append(df)
        
for site_id, analysis_dict in results1:
        results.setdefault(site_id, {}).update(analysis_dict)
        

In [ ]:
dic_timing['GEV approach 2']['regression start'] = datetime.now()

# ------------------------------------------------------------------------------------------

for col_label in ('year', 'sim_year'):
        if col_label in results[0]['data']:
                break
        else:
                raise ValueError('No year found in data table')

for site_id, dic_location in results.items():
    wls_delta = dic_location['wls_delta']

    fig = dbplt.plot_gev_mu_trend(
            df=dic_location['df'],
            weights=dic_location['weights'],
            year_grid=dic_location['year_grid'],
            year_mean=dic_location['year_mean'],
            y_pred=dic_location['y_pred'],
            wls_delta=wls_delta,
            nonstat_years=dic_location['data'][col_label].values.astype(int), 
            nonstat=dic_location['fit results']['gev_nonstationary'],
            display_results=True,
            colors_reg=colors_reg,
            markers_color=color_marker,
            lat=dic_location['data']['lat'].unique()[0],
            lon=dic_location['data']['lon'].unique()[0],
            site_id=site_id,
    )

    
    if save_regression_summary and ('file location' in dic_location.keys() or 'file_path_report' in dic_location.keys()):
        if 'file location' in dic_location.keys():
            save_path = dic_location['file location']
        else:
            save_path = dic_location['file_path_report']
        with open(save_path + '/WLSdelta_summary.html', 'w') as f:
            f.write( wls_delta.summary().as_html())
        
        lat = str(dic_location['location info']['lat'].round(3))
        lon = str(dic_location['location info']['lon'].round(3))
        file_name = f"/GEVTrendAnalysis_location_{str(site_id)}_{lat}|{lon}.png"
        fig.savefig(save_path+file_name, dpi=300, bbox_inches='tight')

    else:
        print("\t skipping saving GEV μ trend analysis ...")

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 2']['regression end'] = datetime.now()
time_diff = dic_timing['GEV approach 2']['regression end'] - dic_timing['GEV approach 2']['regression start']
print(
    f"\nExecution time for computing regression analysis for annual stationary GEV and non-stationary GEV: "
    f"{time_diff}sec"
    )

In [ ]:
ut.store_analysis_notes(dic_notes_analysis, path_export)

## Future Return Levels based on Regression for LocationParameter

#### Utils

In [12]:
def project_gev_params_to_year(gev_nonstat, year, year_ref):
    """
    Project non-stationary GEV parameters to a given year.
    Assumes linear trend in location only.
    """
    
    mu0, mu1 = gev_nonstat['mu0'], gev_nonstat['mu1']
    sigma, xi = gev_nonstat['sigma'], gev_nonstat['xi']

    mu_t = mu0 + mu1 * (year - year_ref)

    return {
        'mu': mu_t,
        'sigma': sigma,
        'xi': xi
    }


def return_levels_at_year(gev_nonstat, return_periods, year, year_ref):
    """
    Return levels for a non-stationary GEV evaluated at a specific year.
    """
    params = project_gev_params_to_year(gev_nonstat, year, year_ref)

    p = 1 - 1 / array(return_periods)

    rl = genextreme.ppf(
        p,
        c=-params['xi'],
        loc=params['mu'],
        scale=params['sigma']
    )

    return dict(zip(return_periods, rl))


def return_level_ci(gev_nonstat, T, year, cov_matrix, year_ref, alpha=0.05):

    mu0, mu1 = gev_nonstat['mu0'], gev_nonstat['mu1']
    sigma, xi = gev_nonstat['sigma'], gev_nonstat['xi']

    var_mu0 = cov_matrix.loc['mu0','mu0']
    var_mu1 = cov_matrix.loc['mu1','mu1']
    cov_mu0_mu1 = cov_matrix.loc['mu0','mu1']

    t = year - year_ref
    var_mu_t = var_mu0 + t**2 * var_mu1 + 2*t*cov_mu0_mu1
    sd_mu_t = sqrt(var_mu_t)

    zT = mu0 + mu1*t + (sigma/xi) * ((-log(1-1/T))**(-xi)-1)

    zT_lower = zT - norm.ppf(1-alpha/2)*sd_mu_t
    zT_upper = zT + norm.ppf(1-alpha/2)*sd_mu_t

    return zT, (zT_lower, zT_upper)


def ensemble_return_levels(models_results, T, year, year_ref):
    zT_list = []
    zT_lower_list = []
    zT_upper_list = []

    for res in models_results:
        gev = res['gev_nonstationary']
        cov = res['cov_matrix']
        zT, (zl, zu) = return_level_ci(gev, T, year, cov, year_ref)
        zT_list.append(zT)
        zT_lower_list.append(zl)
        zT_upper_list.append(zu)

    ensemble_mean = mean(zT_list)
    ensemble_lower = min(zT_lower_list)
    ensemble_upper = max(zT_upper_list)

    return ensemble_mean, (ensemble_lower, ensemble_upper)


#### (Re-) import data and results 
Selecting a subset of 10 locations

In [23]:
year_future_risk = 2050

In [10]:
path_results = '../output/gev_analysis/pooled/2026-02-06/'

In [11]:
results = ut.load_fit_results(path_results)

results = ut.select_allowed_locations(dic_data_per_location=results, start_loc=0, end_loc=10)
results.keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

##### Compute Future Risk

In [24]:
rl_2050 = return_levels_at_year(
    results[0]['fit results']['gev_nonstationary'],
    return_periods=[10, 20, 50, 100],
    year=year_future_risk,
    year_ref=results[0]['data'].sim_year.mean()
)

rl_2050

{10: np.float64(0.17764321416255155),
 20: np.float64(0.19465802915308528),
 50: np.float64(0.21476470146978707),
 100: np.float64(0.2285314988863179)}

In [26]:
results_location = results[0]

In [27]:
results_location['fit results']['gev_stationary'].keys()

dict_keys(['shape', 'location', 'scale', 'n_obs', 'log_likelihood', 'aic', 'bic', 'dist_type', 'tail_behavior', 'analysis_per_year', 'return_levels_per_year'])

In [28]:
results_location['fit results']['gev_nonstationary'].keys()

dict_keys(['mu0', 'mu1', 'sigma', 'xi', 'trend_in', 'n_obs', 'log_likelihood', 'aic', 'bic', 'years_mean', 'years_std', 'CI'])

In [ ]:
from typing import Optional
import func_plotting as dbplt
import func_utils as ut
import statsmodels.api as sm
from numpy import (any, array, exp, finfo, float64, full_like, generic, inf,
                   isfinite, isnan, linalg, linspace, log, nan, ndarray,
                   ones_like, sqrt, sum, vstack, zeros)
from pandas import DataFrame, to_numeric
from scipy import optimize, stats
from scipy.optimize import approx_fprime, minimize
from scipy.stats import norm

def compute_cov_matrix(gev_params: dict, data: ndarray, years: ndarray = None, annual_stat:Optional[bool]=False) -> ndarray:
    """
    Compute the covariance matrix of fitted GEV parameters using numerical Hessian.

    Parameters
    ----------
    gev_params : dict
        Fitted GEV parameters. Should contain:
        - 'shape', 'location', 'scale' for stationary
        - or 'mu0', 'mu1', 'sigma', 'xi' for non-stationary location trend
    data : np.ndarray
        Observed annual maxima used for fitting
    years : np.ndarray, optional
        Years array (required if non-stationary model with trend)

    Returns
    -------
    cov_matrix : np.ndarray
        Covariance matrix of parameters
    """
    
    # Define parameter vector theta and negative log-likelihood
    if 'trend_in' in gev_params and gev_params['trend_in'] == 'location':
        # Non-stationary μ(t) = μ0 + μ1 * t
        t = (years - gev_params['years_mean']) / gev_params['years_std']

        def neg_loglik(theta):
            mu0, mu1, sigma, xi = theta
            mu_t = mu0 + mu1 * t
            c = -xi  # SciPy convention
            return -sum(stats.genextreme.logpdf(data, c, loc=mu_t, scale=sigma))
        theta_hat = array([gev_params['mu0'], gev_params['mu1'], gev_params['sigma'], gev_params['xi']])
    elif 
    else:
        # Stationary GEV: μ, σ, ξ
        def neg_loglik(theta):
            xi, mu, sigma = theta
            c = -xi
            return -sum(stats.genextreme.logpdf(data, c, loc=mu, scale=sigma))

        theta_hat = array([gev_params['shape'], gev_params['location'], gev_params['scale']])

    epsilon = sqrt(finfo(float).eps)
    def grad(theta):
        return optimize.approx_fprime(theta, neg_loglik, epsilon)

    H = optimize.approx_fprime(theta_hat, grad, epsilon)

    # Covariance = inverse of Hessian
    try:
        cov_matrix = linalg.inv(H)
    except linalg.LinAlgError:
        logger.info("Warning: Hessian not invertible; returning None")
        return None
    
    return cov_matrix



In [69]:
# annual stat
cov_annual_stationary = compute_cov_matrix(
    gev_params=results_location['fit results']['gev_stationary'], 
    data=results_location['data']['storm_surge']
    )

# non-stat
cov_nonstat = compute_cov_matrix(
    gev_params=results_location['fit results']['gev_nonstationary'], 
    data=results_location['data']['storm_surge'], 
    years=results_location['data']['sim_year']
    )

# stationary
cov_stationary = compute_cov_matrix(
    gev_params=results_location['fit results']['gev_stationary'], 
    data=results_location['data']['storm_surge']
    )

#gev.calculate_return_level(gev_params, return_periods, year=year_future_risk)

In [72]:
results_location['fit results']['gev_nonstationary']

{'mu0': np.float64(0.12555778975653353),
 'mu1': np.float64(-0.00019261132856393816),
 'sigma': np.float64(0.031532579807195135),
 'xi': np.float64(-0.11050816819700307),
 'trend_in': 'location',
 'n_obs': 8248,
 'log_likelihood': np.float64(16115.935726063277),
 'aic': np.float64(-32223.871452126554),
 'bic': np.float64(-32195.800548023766),
 'years_mean': np.float64(1994.2438166828322),
 'years_std': np.float64(16.65194959739354),
 'CI': {'mu_pred': array([0.12594232, 0.12594232, 0.12594232, ..., 0.12519047, 0.12519047,
         0.12519047], shape=(8248,)),
  'mu_lower': array([0.12433591, 0.12433591, 0.12433591, ..., 0.12366713, 0.12366713,
         0.12366713], shape=(8248,)),
  'mu_upper': array([0.12754872, 0.12754872, 0.12754872, ..., 0.12671381, 0.12671381,
         0.12671381], shape=(8248,))}}

In [71]:
results_location['fit results']['gev_stationary']['analysis_per_year']

,shape,location,scale,n_obs,log_likelihood,aic,bic,dist_type,tail_behavior
1961,-0.12198,0.143399,0.0246,12,26.31016,-46.620321,-45.165601,Weibull (Type III),Light (bounded)
1962,-0.231304,0.135892,0.030587,24,48.90198,-91.803961,-88.269799,Weibull (Type III),Light (bounded)
1963,-0.068825,0.129697,0.03463,36,65.686738,-125.373476,-120.622919,Weibull (Type III),Light (bounded)
1964,-0.120335,0.132325,0.035879,46,83.568526,-161.137053,-155.651129,Weibull (Type III),Light (bounded)
1965,-0.241379,0.128805,0.034044,57,110.286615,-214.57323,-208.444076,Weibull (Type III),Light (bounded)
...,...,...,...,...,...,...,...,...,...
2022,-0.067356,0.124734,0.026403,89,186.595596,-367.191193,-359.725284,Weibull (Type III),Light (bounded)
2023,0.750492,0.121732,0.03243,76,143.410329,-280.820659,-273.828459,Fréchet (Type II),Heavy (polynomial)
2024,-0.234376,0.128985,0.030111,60,123.504868,-241.009736,-234.726703,Weibull (Type III),Light (bounded)
2025,0.750412,0.114779,0.035692,46,84.706357,-163.412715,-157.926791,Fréchet (Type II),Heavy (polynomial)


## Time-Varying Exceedance Probability

<i> What is the probability in 2050 of exceeding the 1960 50-year level? <i>

In [ ]:
def exceedance_probability(gev_nonstat, threshold, year, year_ref):
    params = project_gev_params_to_year(gev_nonstat, year, year_ref)

    cdf = genextreme.cdf(
        threshold,
        c=-params['xi'],
        loc=params['mu'],
        scale=params['sigma']
    )

    return 1 - cdf


In [ ]:
z50_1960 = rl_nonstat_start[50] 

p_2050 = exceedance_probability(
    gev_nonstat_loc,
    threshold=z50_1960,
    year=2050,
    year_ref=years.mean()
)


In [ ]:
def analyze_per_location_future(
    data_hindcast,
    site_id,
    lat,
    lon,
    location_info,
    return_periods,
    future_years=(2030, 2050)
):
    results, notes = analyze_per_location(
        data_hindcast,
        site_id,
        lat,
        lon,
        location_info,
        return_periods
    )

    gev_nonstat = results['fit results']['gev_nonstationary']
    years = results['data']['year'].values
    year_ref = years.mean()

    future_rl = {}
    future_probs = {}

    if gev_nonstat:
        # reference level: 50y at start
        z50_ref = results['return_levels']['nonstationary_start']['values'][50]

        for y in future_years:
            future_rl[y] = return_levels_at_year(
                gev_nonstat, return_periods, y, year_ref
            )

            future_probs[y] = exceedance_probability(
                gev_nonstat, z50_ref, y, year_ref
            )

    results['future'] = {
        'return_levels': future_rl,
        'exceedance_probabilities': future_probs,
        'reference_level': z50_ref
    }

    return results, notes


## Multi-model Comparison

Alternative: sample μ0, β from all models → compute return level distribution → percentile CI

Then you can plot spatial maps of ensemble mean & spread across 11,000 locations

# Visualizations

- Time-varying return levels (line + shaded CI)
- Time-varying exceedance probability of historical extremes (line + CI)
- Ensemble mean return levels at future years (bar + error bars)
- Spatial maps (Europe-wide):
    - Mean RL at 2050
    - Change in probability vs baseline (1960)
    - Spread across models

### Return Level Evolution

In [ ]:
def plot_return_level_evolution(gev_nonstat, return_periods, years, year_ref):
    import matplotlib.pyplot as plt

    for T in return_periods:
        rl_t = [
            return_levels_at_year(gev_nonstat, [T], y, year_ref)[T]
            for y in years
        ]
        plt.plot(years, rl_t, label=f'{T}-year')

    plt.xlabel('Year')
    plt.ylabel('Storm surge (m)')
    plt.legend()
    plt.title('Time-varying return levels')
    plt.grid(True)


### Probability Amplification

In [ ]:
def plot_probability_change(gev_nonstat, threshold, years, year_ref):
    import matplotlib.pyplot as plt

    probs = [
        exceedance_probability(gev_nonstat, threshold, y, year_ref)
        for y in years
    ]

    plt.plot(years, probs)
    plt.ylabel('Annual exceedance probability')
    plt.xlabel('Year')
    plt.title('Increasing exceedance probability of historical extreme')
    plt.grid(True)



#### Potential Additional Visualizations
- Maps of 100-year return levels along European coastline
- Difference maps: non-stationary minus stationary → climate change impact
- Probability exceedance curves for selected cities
- Histograms / density of return levels → compare regions
- Time series of non-stationary μ or return levels → show increasing trends